# Force & torque validation suite

Fundamental electromagnetic **force/torque identities** in pure Python, computed with the extractors in `radia_mcp.radia_ngsolve.force`. Each section reconciles an independent route to the same force or torque (coenergy gradient, Maxwell stress, Lorentz law, virtual work) and prints the executed agreement checks. This is the analytic reference layer the FEM/BEM force post-processing (eggshell / Maxwell-surface) is validated against.

*Validation-class corpus kept at `validation_test/force_validation/`; this notebook is the rendered showcase.*

## 1. Coenergy / torque angle-table consistency

From an energy/coenergy table over rotor angle, `tau = dW'/dtheta`. This checks that the differentiated coenergy table reproduces the directly-tabulated torque.

In [1]:
import sys, os
from pathlib import Path
__file__ = str(Path(os.getcwd()).resolve() / "validation_test" / "force_validation" / "validation_coenergy_torque_table_consistency.py")
sys.path.insert(0, str(Path(__file__).resolve().parent))
sys.modules.pop("result_metadata", None)
sys.argv = ["notebook"]

"""Validation-class coenergy/torque angle-table consistency check.

Run:

    python validation_test/force_validation/validation_coenergy_torque_table_consistency.py

This example treats a torque-angle table and a coenergy-angle table as two
views of the same fixed-current virtual-work calculation.  The coenergy includes
a nonperiodic work term, ``T_mean * theta``, so a nonzero mean torque is allowed:

    W'(theta) = T_mean theta + (T_ripple / n) (1 - cos(n theta))
    T(theta) = dW'/dtheta = T_mean + T_ripple sin(n theta)

The validation summary differentiates the coenergy table, compares the selected
central-difference rows against the torque table, and records the integrated
work consistency.
"""

import argparse
import json
import math
import sys
from pathlib import Path


HERE = Path(__file__).resolve().parent
REPO = HERE.parents[1]
SRC = REPO / "packages" / "radia-mcp" / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from radia_mcp.radia_ngsolve.force import coenergy_torque_table_consistency_summary  # noqa: E402
from result_metadata import add_result_metadata  # noqa: E402


OUT_JSON = HERE / "validation_coenergy_torque_table_consistency_summary.json"
SAMPLES = 181


def _tables() -> tuple[list[float], list[float], list[float]]:
    mean_torque_Nm = 10.0
    ripple_torque_Nm = 0.5
    ripple_order = 3
    period = 2.0 * math.pi
    angles = [period * index / (SAMPLES - 1) for index in range(SAMPLES)]
    coenergy = [
        mean_torque_Nm * theta
        + (ripple_torque_Nm / ripple_order) * (1.0 - math.cos(ripple_order * theta))
        for theta in angles
    ]
    torque = [
        mean_torque_Nm + ripple_torque_Nm * math.sin(ripple_order * theta)
        for theta in angles
    ]
    return angles, coenergy, torque


def build_summary() -> dict[str, object]:
    angles, coenergy, torque = _tables()
    consistency = coenergy_torque_table_consistency_summary(
        angles,
        coenergy,
        torque,
        periodic=False,
        torque_abs_tolerance_Nm=2.0e-3,
        torque_rel_tolerance=3.0e-4,
        comparison_stencils=("central",),
    )

    checks = {
        "samples": consistency["n_samples"],
        "reference_checked_count": consistency["reference_checked_count"],
        "status": consistency["status"],
        "max_torque_abs_error_Nm": consistency["max_torque_abs_error_Nm"],
        "max_torque_rel_error": consistency["max_torque_rel_error"],
        "coenergy_delta_J": consistency["coenergy_delta_J"],
        "reference_torque_trapezoid_work_J": consistency["reference_torque_trapezoid_work_J"],
        "reference_work_minus_coenergy_delta_J": consistency["reference_work_minus_coenergy_delta_J"],
    }

    assert checks["samples"] == SAMPLES
    assert checks["reference_checked_count"] == SAMPLES - 2
    assert checks["status"] == "ok"
    assert checks["max_torque_abs_error_Nm"] < 2.0e-3
    assert abs(checks["reference_work_minus_coenergy_delta_J"]) < 1.0e-12

    return {
        "kind": "coenergy_torque_table_consistency_validation",
        "validation_class": True,
        "learning_theme": (
            "torque-angle tables and coenergy-angle tables should agree under "
            "the fixed-current virtual-work derivative"
        ),
        "checks": checks,
        "angles_rad": angles,
        "coenergy_J": coenergy,
        "torque_Nm": torque,
        "consistency": consistency,
    }


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--out", type=Path, default=OUT_JSON)
    args = parser.parse_args()

    summary = add_result_metadata(build_summary(), __file__)
    args.out.parent.mkdir(parents=True, exist_ok=True)
    args.out.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")

    checks = summary["checks"]
    print("[coenergy torque table consistency]")
    print(
        f"  samples={checks['samples']} "
        f"checked={checks['reference_checked_count']} status={checks['status']}"
    )
    print(f"  max_torque_abs_error_Nm={checks['max_torque_abs_error_Nm']:.3e}")
    print(f"  max_torque_rel_error={checks['max_torque_rel_error']:.3e}")
    print(
        "  work_error_J="
        f"{checks['reference_work_minus_coenergy_delta_J']:.3e}"
    )
    print(f"[OK] wrote {args.out}")
    return 0


if __name__ == "__main__":
    main()


[coenergy torque table consistency]
  samples=181 checked=179 status=ok
  max_torque_abs_error_Nm=9.134e-04
  max_torque_rel_error=9.614e-05
  work_error_J=0.000e+00
[OK] wrote \\192.168.11.100\work\00_CAE\Radia\01_GitHub\validation_test\force_validation\validation_coenergy_torque_table_consistency_summary.json


## 2. Maxwell stress contour balance (2D)

The 2D Maxwell-stress line integral around a closed contour must balance segment by segment -- a standard FEM force-extraction post-processing gate.

In [2]:
import sys, os
from pathlib import Path
__file__ = str(Path(os.getcwd()).resolve() / "validation_test" / "force_validation" / "validation_maxwell_contour_segment_balance.py")
sys.path.insert(0, str(Path(__file__).resolve().parent))
sys.modules.pop("result_metadata", None)
sys.argv = ["notebook"]

"""Validation-class 2D Maxwell stress contour balance example.

Run:

    python validation_test/force_validation/validation_maxwell_contour_segment_balance.py

The example integrates a uniform magnetic field around a rectangular closed
contour.  Each segment has a nonzero stress contribution, but the closed
contour's net force cancels to zero.  This is a compact sign/orientation check
before using the same identity on FEM contour data.
"""

import argparse
import json
import sys
from pathlib import Path


HERE = Path(__file__).resolve().parent
REPO = HERE.parents[1]
SRC = REPO / "packages" / "radia-mcp" / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from radia_mcp.radia_ngsolve.force import (  # noqa: E402
    air_gap_maxwell_pressure,
    maxwell_contour_segment_balance_summary_2d,
)
from result_metadata import add_result_metadata  # noqa: E402


OUT_JSON = HERE / "validation_maxwell_contour_segment_balance_summary.json"
CONTOUR = [(-1.0, -0.5), (1.0, -0.5), (1.0, 0.5), (-1.0, 0.5)]
FIELD_T = (1.0, 0.0)


def build_summary() -> dict[str, object]:
    pressure = air_gap_maxwell_pressure(1.0)
    balance = maxwell_contour_segment_balance_summary_2d(
        CONTOUR,
        FIELD_T,
        expected_force_per_depth_N_per_m=(0.0, 0.0),
    )
    checks = {
        "n_segments": balance["n_segments"],
        "polygon_signed_area_m2": balance["polygon_signed_area_m2"],
        "pressure_Pa": pressure,
        "total_force_per_depth_N_per_m": balance["total_force_per_depth_N_per_m"],
        "sum_abs_normal_force_per_depth_N_per_m": balance["sum_abs_normal_force_per_depth_N_per_m"],
        "sum_abs_tangential_force_per_depth_N_per_m": balance["sum_abs_tangential_force_per_depth_N_per_m"],
        "cancellation_ratio": balance["cancellation_ratio"],
        "dominant_segment_index": balance["dominant_segment_index"],
        "status": balance["status"],
    }

    assert checks["n_segments"] == 4
    assert abs(float(checks["polygon_signed_area_m2"]) - 2.0) < 1.0e-12
    assert max(abs(value) for value in checks["total_force_per_depth_N_per_m"]) < 1.0e-9
    assert abs(float(checks["sum_abs_normal_force_per_depth_N_per_m"]) - 6.0 * pressure) < 1.0e-9
    assert abs(float(checks["sum_abs_tangential_force_per_depth_N_per_m"])) < 1.0e-12
    assert abs(float(checks["cancellation_ratio"])) < 1.0e-14
    assert checks["dominant_segment_index"] == 1
    assert checks["status"] == "ok"

    return {
        "kind": "maxwell_contour_segment_balance_validation",
        "validation_class": True,
        "learning_theme": (
            "Closed 2D Maxwell stress contours can have large local segment "
            "forces while the net force cancels by symmetry"
        ),
        "contour_vertices": CONTOUR,
        "uniform_B_T": FIELD_T,
        "checks": checks,
        "balance": balance,
    }


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--out", type=Path, default=OUT_JSON)
    args = parser.parse_args()

    summary = add_result_metadata(build_summary(), __file__)
    args.out.parent.mkdir(parents=True, exist_ok=True)
    args.out.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")

    checks = summary["checks"]
    print("[Maxwell contour segment balance]")
    print(
        f"  segments={checks['n_segments']} area={checks['polygon_signed_area_m2']:.12g} "
        f"status={checks['status']}"
    )
    print(
        f"  net_force={checks['total_force_per_depth_N_per_m']} "
        f"sum_abs_normal={checks['sum_abs_normal_force_per_depth_N_per_m']:.12g}"
    )
    print(f"  cancellation_ratio={checks['cancellation_ratio']:.3e}")
    print(f"[OK] wrote {args.out}")
    return 0


if __name__ == "__main__":
    main()


[Maxwell contour segment balance]
  segments=4 area=2 status=ok
  net_force=[0.0, 0.0] sum_abs_normal=2387324.14638
  cancellation_ratio=0.000e+00
[OK] wrote \\192.168.11.100\work\00_CAE\Radia\01_GitHub\validation_test\force_validation\validation_maxwell_contour_segment_balance_summary.json


## 3. Two-wire Lorentz / virtual-work force

The force between parallel current-carrying wires from the Lorentz law must agree with the virtual-work (coenergy-gradient) result -- including the attraction/repulsion sign.

In [3]:
import sys, os
from pathlib import Path
__file__ = str(Path(os.getcwd()).resolve() / "validation_test" / "force_validation" / "validation_parallel_wire_virtual_work_force.py")
sys.path.insert(0, str(Path(__file__).resolve().parent))
sys.modules.pop("result_metadata", None)
sys.argv = ["notebook"]

"""Validation-class two-wire Lorentz/virtual-work force identity.

This example pins a common magnetostatic post-processing lesson without
depending on any commercial solver:

    Lorentz force on a current filament == d(coenergy)/d(separation)

For two long parallel wires, the separation-dependent mutual coenergy per unit
length is ``-mu0 I1 I2 log(d/d_ref)/(2*pi)``.  Its derivative is the radial
force per unit length.  Like currents therefore have a negative force in the
increasing-separation coordinate, i.e. attraction.

Run:

    python validation_test/force_validation/validation_parallel_wire_virtual_work_force.py
"""

import argparse
import json
import sys
from pathlib import Path


HERE = Path(__file__).resolve().parent
REPO = HERE.parents[1]
SRC = REPO / "packages" / "radia-mcp" / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from radia_mcp.radia_ngsolve.force import parallel_wire_virtual_work_force_summary  # noqa: E402
from result_metadata import add_result_metadata  # noqa: E402


OUT_JSON = HERE / "validation_parallel_wire_virtual_work_force_summary.json"

CASES = [
    {
        "label": "like_currents_horizontal_attract",
        "current1_A": 3.0,
        "current2_A": 2.0,
        "separation_xy_m": (0.035, 0.0),
        "expected_interaction": "attraction",
    },
    {
        "label": "opposite_currents_vertical_repel",
        "current1_A": 3.0,
        "current2_A": -2.0,
        "separation_xy_m": (0.0, 0.035),
        "expected_interaction": "repulsion",
    },
]


def _case_record(case: dict[str, object]) -> dict[str, object]:
    sx, sy = case["separation_xy_m"]
    separation = (sx * sx + sy * sy) ** 0.5
    row = parallel_wire_virtual_work_force_summary(
        case["current1_A"],
        case["current2_A"],
        case["separation_xy_m"],
        displacement_step_m=separation * 1.0e-4,
    )
    return {
        "label": case["label"],
        "expected_interaction": case["expected_interaction"],
        "summary": row,
    }


def build_summary() -> dict[str, object]:
    records = [_case_record(case) for case in CASES]
    max_rel_error = max(record["summary"]["force_rel_error"] for record in records)
    max_abs_error = max(record["summary"]["force_vector_abs_error_N_per_m"] for record in records)
    interactions_ok = all(
        record["summary"]["interaction"] == record["expected_interaction"]
        for record in records
    )
    directions_ok = (
        records[0]["summary"]["virtual_work_radial_force_per_length_N_per_m"] < 0.0
        and records[1]["summary"]["virtual_work_radial_force_per_length_N_per_m"] > 0.0
    )

    checks = {
        "n_cases": len(records),
        "max_force_rel_error": max_rel_error,
        "max_force_vector_abs_error_N_per_m": max_abs_error,
        "interactions_ok": interactions_ok,
        "directions_ok": directions_ok,
        "passed": max_rel_error < 1.0e-8 and interactions_ok and directions_ok,
    }
    assert checks["passed"]

    return {
        "kind": "parallel_wire_virtual_work_force_validation",
        "validation_class": True,
        "force_learning": (
            "fixed-current coenergy derivative and Lorentz force give the same "
            "parallel-wire force per unit length"
        ),
        "checks": checks,
        "cases": records,
    }


def _json_clean(value):
    if isinstance(value, float):
        return 0.0 if value == 0.0 else value
    if isinstance(value, list):
        return [_json_clean(item) for item in value]
    if isinstance(value, tuple):
        return [_json_clean(item) for item in value]
    if isinstance(value, dict):
        return {key: _json_clean(item) for key, item in value.items()}
    return value


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--out", type=Path, default=OUT_JSON)
    args = parser.parse_args()

    summary = add_result_metadata(_json_clean(build_summary()), __file__)
    args.out.parent.mkdir(parents=True, exist_ok=True)
    args.out.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")

    checks = summary["checks"]
    print("[parallel wire virtual work force]")
    print(f"  cases={checks['n_cases']}, max_rel_error={checks['max_force_rel_error']:.3e}")
    for record in summary["cases"]:
        row = record["summary"]
        print(
            f"  {record['label']}: interaction={row['interaction']}, "
            f"radial_virtual={row['virtual_work_radial_force_per_length_N_per_m']:.12g}, "
            f"radial_analytic={row['analytic_radial_force_per_length_N_per_m']:.12g}"
        )
    print(f"[OK] wrote {args.out}")
    return 0


if __name__ == "__main__":
    main()


[parallel wire virtual work force]
  cases=2, max_rel_error=3.333e-09
  like_currents_horizontal_attract: interaction=attraction, radial_virtual=-3.42857144e-05, radial_analytic=-3.42857142857e-05
  opposite_currents_vertical_repel: interaction=repulsion, radial_virtual=3.42857144e-05, radial_analytic=3.42857142857e-05
[OK] wrote \\192.168.11.100\work\00_CAE\Radia\01_GitHub\validation_test\force_validation\validation_parallel_wire_virtual_work_force_summary.json


## 4. Torque waveform comparison (harmonics)

An analytic before/after torque-angle table, compared harmonic by harmonic: mean torque, ripple, and per-order deltas.

In [4]:
import sys, os
from pathlib import Path
__file__ = str(Path(os.getcwd()).resolve() / "validation_test" / "force_validation" / "validation_torque_waveform_comparison.py")
sys.path.insert(0, str(Path(__file__).resolve().parent))
sys.modules.pop("result_metadata", None)
sys.argv = ["notebook"]

"""Validation-class periodic torque waveform comparison.

This example uses an analytic before/after torque-angle table to check that
the comparison summary separates mean torque drift, sample-wise error, and
harmonic ripple changes.

Run:

    python validation_test/force_validation/validation_torque_waveform_comparison.py
"""

import argparse
import json
import math
import sys
from pathlib import Path


HERE = Path(__file__).resolve().parent
REPO = HERE.parents[1]
SRC = REPO / "packages" / "radia-mcp" / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from radia_mcp.radia_ngsolve.solve import torque_angle_sweep_comparison_summary  # noqa: E402
from result_metadata import add_result_metadata  # noqa: E402


OUT_JSON = HERE / "validation_torque_waveform_comparison_summary.json"
SAMPLES = 720


def _waveform(mean_torque_Nm: float, ripple6_Nm: float, ripple12_Nm: float) -> list[float]:
    values = []
    for idx in range(SAMPLES):
        theta = 2.0 * math.pi * idx / SAMPLES
        values.append(
            mean_torque_Nm
            + ripple6_Nm * math.cos(6 * theta)
            + ripple12_Nm * math.sin(12 * theta)
        )
    return values


def _by_order(rows: list[dict[str, float]]) -> dict[int, dict[str, float]]:
    return {int(row["order"]): row for row in rows}


def _assert_close(actual: float, expected: float, atol: float = 1.0e-12) -> float:
    error = abs(actual - expected)
    if error > atol:
        raise AssertionError(f"{actual!r} != {expected!r}; error={error!r}")
    return error


def build_summary() -> dict[str, object]:
    reference = _waveform(mean_torque_Nm=10.0, ripple6_Nm=0.4, ripple12_Nm=0.1)
    candidate = _waveform(mean_torque_Nm=10.2, ripple6_Nm=0.3, ripple12_Nm=0.12)
    comparison = torque_angle_sweep_comparison_summary(
        reference,
        candidate,
        max_harmonic=18,
        reference_label="analytic_reference",
        candidate_label="analytic_candidate",
    )
    rows = _by_order(comparison["harmonic_delta_rows"])

    expected_sample_delta_rms = math.sqrt(0.2 * 0.2 + (0.1 * 0.1 + 0.02 * 0.02) / 2.0)
    expected_delta_ac_rms = math.sqrt((0.1 * 0.1 + 0.02 * 0.02) / 2.0)
    errors = {
        "mean_delta_error_Nm": _assert_close(comparison["mean_delta_Nm"], 0.2),
        "sample_delta_rms_error_Nm": _assert_close(
            comparison["sample_delta_rms_Nm"],
            expected_sample_delta_rms,
        ),
        "delta_ac_rms_error_Nm": _assert_close(
            comparison["difference_summary"]["ac_rms_torque_Nm"],
            expected_delta_ac_rms,
        ),
        "sixth_harmonic_delta_error_Nm": _assert_close(rows[6]["amplitude_delta_Nm"], -0.1),
        "twelfth_harmonic_delta_error_Nm": _assert_close(rows[12]["amplitude_delta_Nm"], 0.02),
    }

    if comparison["dominant_harmonic_changed"]:
        raise AssertionError("dominant harmonic should remain unchanged")
    if comparison["worst_harmonic_order"] != 6:
        raise AssertionError("largest ripple amplitude change should be harmonic order 6")

    checks = {
        "samples": SAMPLES,
        "passed": True,
        "max_abs_error_Nm": max(errors.values()),
        "mean_delta_Nm": comparison["mean_delta_Nm"],
        "sample_delta_rms_Nm": comparison["sample_delta_rms_Nm"],
        "expected_sample_delta_rms_Nm": expected_sample_delta_rms,
        "delta_ac_rms_Nm": comparison["difference_summary"]["ac_rms_torque_Nm"],
        "expected_delta_ac_rms_Nm": expected_delta_ac_rms,
        "dominant_harmonic_changed": comparison["dominant_harmonic_changed"],
        "worst_harmonic_order": comparison["worst_harmonic_order"],
        "sixth_harmonic_amplitude_delta_Nm": rows[6]["amplitude_delta_Nm"],
        "twelfth_harmonic_amplitude_delta_Nm": rows[12]["amplitude_delta_Nm"],
    }

    return {
        "kind": "torque_waveform_comparison_validation",
        "validation_class": True,
        "learning_theme": (
            "periodic torque-angle comparisons should separate mean drift, "
            "sample-wise error, and harmonic ripple deltas"
        ),
        "checks": checks,
        "errors": errors,
        "comparison": comparison,
    }


def _json_clean(value):
    if isinstance(value, float):
        return 0.0 if value == 0.0 else value
    if isinstance(value, list):
        return [_json_clean(item) for item in value]
    if isinstance(value, tuple):
        return [_json_clean(item) for item in value]
    if isinstance(value, dict):
        return {key: _json_clean(item) for key, item in value.items()}
    return value


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--out", type=Path, default=OUT_JSON)
    args = parser.parse_args()

    summary = add_result_metadata(_json_clean(build_summary()), __file__)
    args.out.parent.mkdir(parents=True, exist_ok=True)
    args.out.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")

    checks = summary["checks"]
    print("[torque waveform comparison]")
    print(f"  samples={checks['samples']}, max_abs_error={checks['max_abs_error_Nm']:.3e}")
    print(f"  mean_delta_Nm={checks['mean_delta_Nm']:.12g}")
    print(f"  sample_delta_rms_Nm={checks['sample_delta_rms_Nm']:.12g}")
    print(f"  worst_harmonic_order={checks['worst_harmonic_order']}")
    print(f"  sixth_harmonic_amplitude_delta_Nm={checks['sixth_harmonic_amplitude_delta_Nm']:.12g}")
    print(f"  twelfth_harmonic_amplitude_delta_Nm={checks['twelfth_harmonic_amplitude_delta_Nm']:.12g}")
    print(f"[OK] wrote {args.out}")
    return 0


if __name__ == "__main__":
    main()


[torque waveform comparison]
  samples=720, max_abs_error=1.155e-15
  mean_delta_Nm=0.2
  sample_delta_rms_Nm=0.212602916255
  worst_harmonic_order=6
  sixth_harmonic_amplitude_delta_Nm=-0.1
  twelfth_harmonic_amplitude_delta_Nm=0.02
[OK] wrote \\192.168.11.100\work\00_CAE\Radia\01_GitHub\validation_test\force_validation\validation_torque_waveform_comparison_summary.json


## 5. Torque waveform harmonic health

Mean / ripple / dominant-harmonic audit of a torque waveform -- the health metrics used to flag cogging and ripple regressions.

In [5]:
import sys, os
from pathlib import Path
__file__ = str(Path(os.getcwd()).resolve() / "validation_test" / "force_validation" / "validation_torque_waveform_health.py")
sys.path.insert(0, str(Path(__file__).resolve().parent))
sys.modules.pop("result_metadata", None)
sys.argv = ["notebook"]

"""Validation-class torque waveform harmonic health example.

Run:

    python validation_test/force_validation/validation_torque_waveform_health.py

The example uses an analytic torque-angle waveform with a mean component and
two ripple harmonics.  The health summary separates mean torque, RMS ripple,
dominant harmonic order, and the variance budget of the largest harmonics.
"""

import argparse
import json
import math
import sys
from pathlib import Path


HERE = Path(__file__).resolve().parent
REPO = HERE.parents[1]
SRC = REPO / "packages" / "radia-mcp" / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from radia_mcp.radia_ngsolve.solve import torque_angle_sweep_health_summary  # noqa: E402
from result_metadata import add_result_metadata  # noqa: E402


OUT_JSON = HERE / "validation_torque_waveform_health_summary.json"
SAMPLES = 720


def _waveform(mean_torque_Nm: float, ripple6_Nm: float, ripple12_Nm: float) -> list[float]:
    values = []
    for idx in range(SAMPLES):
        theta = 2.0 * math.pi * idx / SAMPLES
        values.append(
            mean_torque_Nm
            + ripple6_Nm * math.cos(6 * theta)
            + ripple12_Nm * math.sin(12 * theta)
        )
    return values


def _assert_close(actual: float, expected: float, atol: float = 1.0e-12) -> float:
    error = abs(actual - expected)
    if error > atol:
        raise AssertionError(f"{actual!r} != {expected!r}; error={error!r}")
    return error


def build_summary() -> dict[str, object]:
    mean = 10.0
    ripple6 = 0.4
    ripple12 = 0.1
    torque = _waveform(mean, ripple6, ripple12)
    health = torque_angle_sweep_health_summary(
        torque,
        max_harmonic=18,
        max_ac_rms_over_mean=0.04,
        allowed_dominant_harmonics=[6],
        min_mean_abs_torque_Nm=9.0,
        top_harmonics=3,
    )
    top = {row["order"]: row for row in health["top_harmonic_rows"]}
    expected_ac_rms = math.sqrt((ripple6 * ripple6 + ripple12 * ripple12) / 2.0)
    expected_variance6 = ripple6 * ripple6 / (ripple6 * ripple6 + ripple12 * ripple12)
    expected_variance12 = ripple12 * ripple12 / (ripple6 * ripple6 + ripple12 * ripple12)
    errors = {
        "mean_error_Nm": _assert_close(health["mean_torque_Nm"], mean),
        "ac_rms_error_Nm": _assert_close(health["ac_rms_torque_Nm"], expected_ac_rms),
        "sixth_amplitude_error_Nm": _assert_close(top[6]["amplitude_Nm"], ripple6),
        "twelfth_amplitude_error_Nm": _assert_close(top[12]["amplitude_Nm"], ripple12),
        "sixth_variance_fraction_error": _assert_close(top[6]["ac_variance_fraction"], expected_variance6),
        "twelfth_variance_fraction_error": _assert_close(top[12]["ac_variance_fraction"], expected_variance12),
    }
    checks = {
        "samples": SAMPLES,
        "status": health["status"],
        "mean_torque_Nm": health["mean_torque_Nm"],
        "ac_rms_torque_Nm": health["ac_rms_torque_Nm"],
        "ac_rms_over_mean": health["ac_rms_over_mean"],
        "dominant_harmonic": health["dominant_harmonic"],
        "dominant_harmonic_amplitude_Nm": health["dominant_harmonic_amplitude_Nm"],
        "sixth_variance_fraction": top[6]["ac_variance_fraction"],
        "twelfth_variance_fraction": top[12]["ac_variance_fraction"],
        "max_abs_error": max(errors.values()),
    }

    assert checks["status"] == "ok"
    assert checks["dominant_harmonic"] == 6
    assert checks["max_abs_error"] < 1.0e-12

    return {
        "kind": "torque_waveform_health_validation",
        "validation_class": True,
        "learning_theme": (
            "torque-angle tables should separate mean torque, RMS ripple, "
            "dominant harmonic order, and harmonic variance budget"
        ),
        "checks": checks,
        "errors": errors,
        "health": health,
    }


def _json_clean(value):
    if isinstance(value, float):
        return 0.0 if value == 0.0 else value
    if isinstance(value, list):
        return [_json_clean(item) for item in value]
    if isinstance(value, tuple):
        return [_json_clean(item) for item in value]
    if isinstance(value, dict):
        return {key: _json_clean(item) for key, item in value.items()}
    return value


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--out", type=Path, default=OUT_JSON)
    args = parser.parse_args()

    summary = add_result_metadata(_json_clean(build_summary()), __file__)
    args.out.parent.mkdir(parents=True, exist_ok=True)
    args.out.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")

    checks = summary["checks"]
    print("[torque waveform health]")
    print(
        f"  samples={checks['samples']} status={checks['status']} "
        f"mean={checks['mean_torque_Nm']:.12g} ac_rms={checks['ac_rms_torque_Nm']:.12g}"
    )
    print(
        f"  dominant_harmonic={checks['dominant_harmonic']} "
        f"amplitude={checks['dominant_harmonic_amplitude_Nm']:.12g}"
    )
    print(
        f"  variance fractions: h6={checks['sixth_variance_fraction']:.12g} "
        f"h12={checks['twelfth_variance_fraction']:.12g}"
    )
    print(f"[OK] wrote {args.out}")
    return 0


if __name__ == "__main__":
    main()


[torque waveform health]
  samples=720 status=ok mean=10 ac_rms=0.291547594742
  dominant_harmonic=6 amplitude=0.4
  variance fractions: h6=0.941176470588 h12=0.0588235294118
[OK] wrote \\192.168.11.100\work\00_CAE\Radia\01_GitHub\validation_test\force_validation\validation_torque_waveform_health_summary.json


## 6. Virtual-work force sweep audit

Finite-difference of a coenergy displacement sweep against the analytic force, including the fixed-current vs fixed-flux sign gate and the reference error budget.

In [6]:
import sys, os
from pathlib import Path
__file__ = str(Path(os.getcwd()).resolve() / "validation_test" / "force_validation" / "validation_virtual_work_force_sweep_audit.py")
sys.path.insert(0, str(Path(__file__).resolve().parent))
sys.modules.pop("result_metadata", None)
sys.argv = ["notebook"]

"""Validation-class virtual-work force sweep audit.

Run:

    python validation_test/force_validation/validation_virtual_work_force_sweep_audit.py

This example samples the separation-dependent coenergy per unit length of two
long line currents,

    W'(d) = -mu0 I1 I2 log(d/d_ref)/(2*pi),

and audits the finite-difference force table against the analytic radial force

    F(d) = dW'/dd = -mu0 I1 I2/(2*pi*d).

The lesson is the sweep form: solver outputs usually arrive as a table of
energy samples, so the validation object should preserve stencils, curvature,
and reference-force errors rather than only one pass/fail number.
"""

import argparse
import json
import math
import sys
from pathlib import Path


HERE = Path(__file__).resolve().parent
REPO = HERE.parents[1]
SRC = REPO / "packages" / "radia-mcp" / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from radia_mcp.radia_ngsolve.force import (  # noqa: E402
    MU0,
    virtual_work_force_sweep_audit_summary,
)
from result_metadata import add_result_metadata  # noqa: E402


OUT_JSON = HERE / "validation_virtual_work_force_sweep_audit_summary.json"


def _line_current_coenergy_and_force(
    separations_m: list[float],
    current1_A: float,
    current2_A: float,
    reference_separation_m: float,
) -> tuple[list[float], list[float]]:
    coefficient = MU0 * current1_A * current2_A / (2.0 * math.pi)
    coenergy = [
        -coefficient * math.log(distance / reference_separation_m)
        for distance in separations_m
    ]
    radial_force = [-coefficient / distance for distance in separations_m]
    return coenergy, radial_force


def build_summary() -> dict[str, object]:
    current1_A = 3.0
    current2_A = 2.0
    separations_m = [0.020 + 0.001 * index for index in range(13)]
    reference_separation_m = separations_m[len(separations_m) // 2]
    coenergy, reference_force = _line_current_coenergy_and_force(
        separations_m,
        current1_A,
        current2_A,
        reference_separation_m,
    )
    sweep = virtual_work_force_sweep_audit_summary(
        separations_m,
        coenergy,
        energy_kind="coenergy",
        reference_force_N=reference_force,
        force_abs_tolerance_N=1.0e-15,
        force_rel_tolerance=1.5e-3,
        comparison_stencils=("central",),
    )

    checks = {
        "n_samples": sweep["n_samples"],
        "reference_checked_count": sweep["reference_checked_count"],
        "status": sweep["status"],
        "max_reference_force_rel_error": sweep["max_reference_force_rel_error"],
        "force_min_N_per_m": sweep["force_min_N"],
        "force_max_N_per_m": sweep["force_max_N"],
        "max_abs_force_gradient_N_per_m2": sweep["max_abs_force_gradient_N_per_m"],
    }

    assert checks["n_samples"] == len(separations_m)
    assert checks["reference_checked_count"] == len(separations_m) - 2
    assert checks["status"] == "ok"
    assert checks["max_reference_force_rel_error"] < 1.5e-3
    assert checks["force_min_N_per_m"] < checks["force_max_N_per_m"] < 0.0
    assert checks["max_abs_force_gradient_N_per_m2"] > 0.0

    return {
        "kind": "virtual_work_force_sweep_audit_validation",
        "validation_class": True,
        "force_learning": (
            "energy/coenergy sweeps should preserve finite-difference stencils, "
            "force-gradient estimates, and reference-force errors"
        ),
        "current1_A": current1_A,
        "current2_A": current2_A,
        "reference_separation_m": reference_separation_m,
        "checks": checks,
        "separations_m": separations_m,
        "coenergy_J_per_m": coenergy,
        "analytic_radial_force_N_per_m": reference_force,
        "sweep": sweep,
    }


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--out", type=Path, default=OUT_JSON)
    args = parser.parse_args()

    summary = add_result_metadata(build_summary(), __file__)
    args.out.parent.mkdir(parents=True, exist_ok=True)
    args.out.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")

    checks = summary["checks"]
    print("[virtual work force sweep audit]")
    print(
        f"  samples={checks['n_samples']} "
        f"checked={checks['reference_checked_count']} status={checks['status']}"
    )
    print(
        f"  force range=[{checks['force_min_N_per_m']:.12g}, "
        f"{checks['force_max_N_per_m']:.12g}] N/m"
    )
    print(
        "  max_reference_force_rel_error="
        f"{checks['max_reference_force_rel_error']:.3e}"
    )
    print(f"[OK] wrote {args.out}")
    return 0


if __name__ == "__main__":
    main()


[virtual work force sweep audit]
  samples=13 checked=11 status=ok
  force range=[-5.85481970033e-05, -3.80984379775e-05] N/m
  max_reference_force_rel_error=7.569e-04
[OK] wrote \\192.168.11.100\work\00_CAE\Radia\01_GitHub\validation_test\force_validation\validation_virtual_work_force_sweep_audit_summary.json
